# 365 Probabilidades - Dia #153
## Qual a probabilidade de a sexta-feira 13 ser mais perigosa do que qualquer outra sexta?

**Tipo:** Comportamento · Superstição
**Data de publicação:** 2026-11-13
**Ferramenta:** Python
**Decisão analisada:** Vale mudar os planos (pegar estrada, mexer no dinheiro) porque é sexta-feira 13?
**Hashtag:** #365Probabilidades #Dia153

---

### 📖 A História

Dizem que sexta-feira 13 é dia de não tomar decisões. Ouvi isso esses dias e fiquei pensando se faz algum sentido.

Hoje é sexta-feira 13. Tem gente que muda os planos e adia a viagem. Tem gente que não marca nada importante. E tem gente que diz que não acredita, mas prefere não arriscar.

Se a sexta-feira 13 fosse mesmo mais perigosa, isso deveria aparecer nos números. Fui procurar em dois lugares onde o azar pode ser medido com cuidado, nas estradas e na Bolsa.

---

### 📚 O Conceito: comparar a sexta-feira 13 com as sextas vizinhas

Comparar a sexta-feira 13 com a média de todos os dias não serve, porque sexta já é um dia
diferente. Tem mais trânsito, mais viagem e mais gente cansada. O jeito justo é comparar cada
sexta-feira 13 com a sexta da semana anterior e a da semana seguinte. Elas têm o mesmo mês, a
mesma estação, quase o mesmo movimento. A única diferença é o número no calendário.

Foi o que fizeram Igor Radun e Heikki Summala em 2004, na Finlândia, para testar um estudo de
2002 que tinha achado mais mortes de mulheres no trânsito na sexta-feira 13. Com 21 sextas-feiras
13 e as sextas vizinhas, sem feriados, eles não acharam diferença nenhuma.

A fama vem de mais longe. Em 1993, um artigo na edição de Natal do *BMJ*, conhecida pelos textos
bem-humorados, relatou até 52% mais internações por acidente de trânsito na sexta-feira 13, num
único hospital inglês e com poucas sextas.

Na Bolsa a história é parecida. Em 1987, Robert Kolb e Ricardo Rodriguez acharam retornos menores
na sexta-feira 13 nos Estados Unidos. No ano seguinte, com uma série mais longa, outros autores
acharam o contrário, sem significância, e concluíram que tinha sido acaso.

---

### 🧮 O Modelo

**Fontes:**
- **PRF, Dados Abertos, acidentes agrupados por ocorrência, 2015 a 2025.** Todos os acidentes
  registrados nas rodovias federais, com data, mortos e feridos graves. Dados baixados por Ana
  em 04/11/2026 e salvos em `dados153/`.
- **Ibovespa (^BVSP), fechamento diário de 2000 a 2026,** série do Yahoo Finance baixada por Ana
  em 04/11/2026 (`dados153/ibovespa_diario.csv`).
- Radun, I. & Summala, H. (2004). Females do not have more injury road accidents on Friday the
  13th. *BMC Public Health*, 4, 54. Finlândia, 1989-2002, 21 sextas-feiras 13 pareadas com as
  sextas vizinhas, sem feriados. Sem diferença para mulheres nem para homens.
- Näyhä, S. (2002). Traffic deaths and superstition on Friday the 13th. *American Journal of
  Psychiatry*, 159, 2110-2111.
- Scanlon, T. J., Luben, R. N., Scanlon, F. L. & Singleton, N. (1993). Is Friday the 13th bad
  for your health? *BMJ*, 307, 1584-1586. Edição de Natal, um hospital, poucas sextas.
- Kolb, R. W. & Rodriguez, R. J. (1987). Friday the thirteenth: "Part VII", a note. *Journal of
  Finance*, 42(5), 1385-1387. Dados CRSP 1962-1985, retornos menores na sexta-feira 13.
- Dyl, E. A. & Maberly, E. D. (1988), resposta com o S&P 500 de 1940 a 1987: retornos um pouco
  maiores, sem significância. Lucey, B. M. (2001). *Applied Economics Letters*, 8(9), 577-579:
  19 países, alguns indícios de retornos maiores.

**Desenho.** Nas estradas, cada sexta-feira 13 é comparada com a sexta 7 dias antes e a sexta 7
dias depois. Saem da conta as sextas que são feriado nacional, Sexta-feira Santa, sexta de
Carnaval, sexta de ponte (depois de feriado na quinta), sexta antes de feriado na segunda e as
vésperas de Natal e Ano-Novo. Se uma vizinha cai fora, a comparação usa só a outra. Também saem
os dias críticos da pandemia, de 1º de março a 30 de junho de 2020. Na Bolsa, sai o mesmo
período da pandemia.

**Família do teste.**
- Razão de taxas (sexta-feira 13 contra vizinhas) por regressão de Poisson com um efeito fixo
  para cada trio de sextas e erro corrigido para a dispersão extra (quase-Poisson). IC 95% e
  P(razão > 1).
- Duelo por sexta: em quantas sextas-feiras 13 houve mais acidentes do que na média das vizinhas,
  com posterior Beta de Jeffreys.
- Teste placebo: a mesma conta para as sextas de todos os outros dias do mês (dia 1, dia 2, até
  dia 31), para ver onde o 13 fica.
- Bolsa: diferença de retorno médio com IC 95% por bootstrap e proporção de dias de alta com
  posteriores Beta de Jeffreys e Monte Carlo.

**Sem ×0,80.** São registros administrativos e preços de mercado, não pesquisa autorrelatada.


In [ ]:
import zipfile, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
from dateutil.easter import easter

SEMENTE = 153
rng = np.random.default_rng(SEMENTE)
SORTEIOS = 200_000
PASTA = 'dados153'

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE_PRF = 'Fonte: PRF, Dados Abertos, acidentes por ocorrência 2015-2025 · cálculo próprio | #365Probabilidades'
RODAPE_BOLSA = 'Fonte: Ibovespa (^BVSP), Yahoo Finance, 2000-2026 · cálculo próprio | #365Probabilidades'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(x):
    return f"{x:,.0f}".replace(",", ".")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

print("✅ Bibliotecas carregadas · semente", SEMENTE)


In [ ]:
# --- DADOS ---

# 1. Acidentes nas rodovias federais (PRF), um zip por ano
partes = []
for arq in sorted(glob.glob(os.path.join(PASTA, 'datatran20*.zip'))):
    z = zipfile.ZipFile(arq)
    d = pd.read_csv(z.open(z.namelist()[0]), sep=';', encoding='latin1', dtype=str,
                    usecols=['id', 'data_inversa', 'mortos', 'feridos_graves'])
    s = d['data_inversa'].str.strip()
    data = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    for fmt in ('%d/%m/%Y', '%d/%m/%y'):
        data = data.fillna(pd.to_datetime(s, format=fmt, errors='coerce'))
    d['data'] = data
    partes.append(d)
acid = pd.concat(partes, ignore_index=True)
acid = acid.dropna(subset=['data']).drop_duplicates('id')
for c in ['mortos', 'feridos_graves']:
    acid[c] = pd.to_numeric(acid[c], errors='coerce').fillna(0)
dia = acid.groupby('data').agg(acidentes=('id', 'size'), mortos=('mortos', 'sum'),
                               feridos_graves=('feridos_graves', 'sum'))

# 2. Ibovespa diário
ibov = pd.read_csv(os.path.join(PASTA, 'ibovespa_diario.csv'), parse_dates=['data']).sort_values('data')
ibov = ibov.drop_duplicates('data')
ibov['retorno'] = np.log(ibov['fechamento']).diff() * 100
ibov = ibov.dropna(subset=['retorno'])

# 3. Calendário de exclusões
def feriados_nacionais(ano):
    pascoa = pd.Timestamp(easter(ano))
    fixos = [(1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)]
    if ano >= 2024:
        fixos.append((11, 20))
    f = {pd.Timestamp(ano, m, d) for m, d in fixos}
    f |= {pascoa - pd.Timedelta(days=2),            # Sexta-feira Santa
          pascoa - pd.Timedelta(days=48),           # segunda de Carnaval
          pascoa - pd.Timedelta(days=47),           # terça de Carnaval
          pascoa + pd.Timedelta(days=60)}           # Corpus Christi
    return f

FERIADOS = set().union(*[feriados_nacionais(a) for a in range(1999, 2028)])
PANDEMIA = (pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'))

EXCLUIR = True   # False refaz a conta sem tirar feriados nem pandemia (teste de sensibilidade)

def motivo_exclusao(x):
    if not EXCLUIR:
        return None
    if PANDEMIA[0] <= x <= PANDEMIA[1]:
        return 'pandemia'
    if x in FERIADOS:
        return 'feriado'
    if x == pd.Timestamp(easter(x.year)) - pd.Timedelta(days=51):
        return 'sexta de Carnaval'
    if x - pd.Timedelta(days=1) in FERIADOS:
        return 'ponte'
    if x + pd.Timedelta(days=3) in FERIADOS:
        return 'véspera de feriado prolongado'
    if (x.month, x.day) in [(12, 24), (12, 31)]:
        return 'véspera de Natal ou Ano-Novo'
    return None

print("=" * 72)
print("  DADOS")
print("=" * 72)
print(f"  PRF: {mil(len(acid))} acidentes, de {dia.index.min():%d/%m/%Y} a {dia.index.max():%d/%m/%Y}")
print(f"       {mil(acid['mortos'].sum())} mortos · {mil(acid['feridos_graves'].sum())} feridos graves")
print(f"  Ibovespa: {mil(len(ibov))} pregões, de {ibov['data'].min():%d/%m/%Y} a {ibov['data'].max():%d/%m/%Y}")

sextas13 = [x for x in dia.index if x.day == 13 and x.weekday() == 4]
print(f"\n  Sextas-feiras 13 no período da PRF: {len(sextas13)}")
for x in sextas13:
    m = motivo_exclusao(x)
    print(f"   {x:%d/%m/%Y}  {'fica' if m is None else 'sai (' + m + ')'}")
print("=" * 72)


In [ ]:
# --- O MODELO ---

def montar_trios(dia_do_mes):
    linhas = []
    for x in dia.index:
        if x.day != dia_do_mes or x.weekday() != 4 or motivo_exclusao(x):
            continue
        viz = [x + pd.Timedelta(days=k) for k in (-7, 7)]
        viz = [v for v in viz if v in dia.index and motivo_exclusao(v) is None]
        if not viz:
            continue
        linhas.append({'trio': x, 'data': x, 'alvo': 1, **dia.loc[x].to_dict()})
        for v in viz:
            linhas.append({'trio': x, 'data': v, 'alvo': 0, **dia.loc[v].to_dict()})
    return pd.DataFrame(linhas)

def razao(trios, var):
    if trios['alvo'].sum() < 2:
        return np.nan, np.nan, np.nan, np.nan
    # Poisson com efeito fixo por trio, resolvido pela verossimilhança condicional:
    # dado o total do trio, a sexta 13 recebe uma fração RR / (RR + k), k = número de vizinhas
    g = trios.groupby('trio')
    y13 = g.apply(lambda t: t.loc[t['alvo'] == 1, var].sum()).values.astype(float)
    tot = g[var].sum().values.astype(float)
    k = g['alvo'].apply(lambda a: (a == 0).sum()).values.astype(float)
    def nll(b):
        q = np.exp(b) / (np.exp(b) + k)
        return -(y13 * np.log(q) + (tot - y13) * np.log(1 - q)).sum()
    b = minimize_scalar(nll, bounds=(-3, 3), method='bounded').x
    q = np.exp(b) / (np.exp(b) + k)
    info = (tot * q * (1 - q)).sum()
    # dispersão extra (quase-Poisson): Pearson X2 / graus de liberdade
    mu13, muv = tot * q, tot * (1 - q) / k
    x2 = ((y13 - mu13) ** 2 / mu13).sum()
    for (_, t), mv in zip(g, muv):
        x2 += ((t.loc[t['alvo'] == 0, var].values - mv) ** 2 / mv).sum()
    gl = len(trios) - (len(tot) + 1)
    phi = x2 / gl if gl > 0 else 1.0
    se = np.sqrt(phi / info)
    return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), stats.norm.sf(0, loc=b, scale=se)

T13 = montar_trios(13)

# CAMADA 1 · Sexta-feira 13 contra as sextas vizinhas
print("=" * 72)
print("  CAMADA 1 · SEXTA-FEIRA 13 CONTRA AS SEXTAS VIZINHAS (sem feriados nem pandemia)")
print("=" * 72)
alvo = T13[T13['alvo'] == 1].set_index('trio')
viz = T13[T13['alvo'] == 0].groupby('trio')[['acidentes', 'mortos', 'feridos_graves']].mean()
COMP = alvo[['acidentes', 'mortos', 'feridos_graves']].join(viz, rsuffix='_viz')
for x, r in COMP.iterrows():
    print(f"  {x:%d/%m/%Y}  acidentes {int(r.acidentes):>4} contra {vir(r.acidentes_viz):>6}"
          f"   mortos {int(r.mortos):>3} contra {vir(r.mortos_viz):>5}")
N13 = len(COMP)
print(f"\n  Sextas-feiras 13 na conta: {N13}")

RR = {}
for var, nome in [('acidentes', 'Acidentes'), ('feridos_graves', 'Feridos graves'), ('mortos', 'Mortos')]:
    rr, lo, hi, p = razao(T13, var)
    RR[nome] = (rr, lo, hi, p)
    print(f"  {nome:<15} razão {vir(rr, 2)}  IC 95% [{vir(lo, 2)}; {vir(hi, 2)}]   P(razão > 1) = {vir(p * 100)}%")

EXCLUIR = False
T_TUDO = montar_trios(13)
print(f"\n  Sensibilidade, sem tirar feriados nem pandemia ({int(T_TUDO['alvo'].sum())} sextas-feiras 13):")
for var, nome in [('acidentes', 'Acidentes'), ('mortos', 'Mortos')]:
    rr, lo, hi, p = razao(T_TUDO, var)
    print(f"  {nome:<15} razão {vir(rr, 2)}  IC 95% [{vir(lo, 2)}; {vir(hi, 2)}]   P(razão > 1) = {vir(p * 100)}%")
EXCLUIR = True

# CAMADA 2 · Duelo sexta a sexta
VENCE = int((COMP['acidentes'] > COMP['acidentes_viz']).sum())
post_duelo = jeffreys(VENCE, N13)
dlo, dhi = post_duelo.interval(0.95)
print("\n" + "=" * 72)
print("  CAMADA 2 · DUELO SEXTA A SEXTA")
print("=" * 72)
print(f"  A sexta-feira 13 teve mais acidentes que a média das vizinhas em {VENCE} de {N13}")
print(f"  Posterior de Jeffreys: {vir(post_duelo.mean() * 100)}%  IC 95% [{vir(dlo * 100)}; {vir(dhi * 100)}]")
print(f"  P(a sexta-feira 13 perde o duelo mais da metade das vezes) = {vir(post_duelo.cdf(0.5) * 100)}%")

# CAMADA 3 · Placebo, todos os dias do mês
print("\n" + "=" * 72)
print("  CAMADA 3 · PLACEBO · A MESMA CONTA PARA AS SEXTAS DE CADA DIA DO MÊS")
print("=" * 72)
plac = []
for d in range(1, 32):
    t = montar_trios(d)
    if t.empty:
        continue
    rr_a = razao(t, 'acidentes')[0]
    rr_m = razao(t, 'mortos')[0]
    plac.append({'dia': d, 'rr_acidentes': rr_a, 'rr_mortos': rr_m, 'sextas': int(t['alvo'].sum())})
PLAC = pd.DataFrame(plac).sort_values('dia').reset_index(drop=True)
POS13 = {}
for v in ['rr_acidentes', 'rr_mortos']:
    ordem = PLAC.sort_values(v, ascending=False)['dia'].tolist()
    POS13[v] = ordem.index(13) + 1
print(PLAC.round(3).to_string(index=False))
print(f"\n  Acidentes: a sexta 13 fica em {POS13['rr_acidentes']}º lugar entre {len(PLAC)} dias do mês")
print(f"  Mortos:    a sexta 13 fica em {POS13['rr_mortos']}º lugar entre {len(PLAC)} dias do mês")
ACIMA = int((PLAC['rr_mortos'] >= PLAC.loc[PLAC['dia'] == 13, 'rr_mortos'].iloc[0] - 0.005).sum()) - 1
print(f"  Outros dias do mês com razão de mortos igual ou maior que a do 13: {ACIMA}")

# CAMADA 4 · A Bolsa
print("\n" + "=" * 72)
print("  CAMADA 4 · A BOLSA · IBOVESPA NAS SEXTAS-FEIRAS")
print("=" * 72)
sx = ibov[(ibov['data'].dt.weekday == 4) & ~ibov['data'].between(*PANDEMIA)].copy()
sx['treze'] = sx['data'].dt.day == 13
r13, rout = sx.loc[sx['treze'], 'retorno'].values, sx.loc[~sx['treze'], 'retorno'].values
boot = np.array([rng.choice(r13, len(r13)).mean() - rng.choice(rout, len(rout)).mean() for _ in range(20_000)])
DIF, DLO, DHI = r13.mean() - rout.mean(), *np.percentile(boot, [2.5, 97.5])
A13, AOUT = int((r13 > 0).sum()), int((rout > 0).sum())
p13, pout = jeffreys(A13, len(r13)), jeffreys(AOUT, len(rout))
s13, sout = p13.rvs(SORTEIOS, random_state=rng), pout.rvs(SORTEIOS, random_state=rng)
P_PIOR = (s13 < sout).mean()
print(f"  Sextas-feiras 13 com pregão (sem pandemia): {len(r13)} · outras sextas: {mil(len(rout))}")
print(f"  Retorno médio: {vir(r13.mean(), 2)}% contra {vir(rout.mean(), 2)}%")
print(f"  Diferença: {vir(DIF, 2)} ponto  IC 95% bootstrap [{vir(DLO, 2)}; {vir(DHI, 2)}]")
print(f"  Mediana: {vir(np.median(r13), 2)}% contra {vir(np.median(rout), 2)}%")
print(f"  Dias de alta: {A13} de {len(r13)} ({vir(A13 / len(r13) * 100)}%) contra {vir(AOUT / len(rout) * 100)}%")
print(f"  P(a sexta-feira 13 sobe menos vezes que as outras sextas) = {vir(P_PIOR * 100)}%")
sx_tudo = ibov[ibov['data'].dt.weekday == 4]
t13 = sx_tudo.loc[sx_tudo['data'].dt.day == 13, 'retorno']
print(f"\n  Sensibilidade, com a pandemia: {len(t13)} sextas-feiras 13, retorno médio {vir(t13.mean(), 2)}%,"
      f" alta em {vir((t13 > 0).mean() * 100)}% (13/03/2020 sozinho: {vir(t13[sx_tudo['data'] == '2020-03-13'].iloc[0], 1)}%)")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · Cada sexta-feira 13 e suas vizinhas
fig1, ax1 = plt.subplots(figsize=(12, 7.5))
y = np.arange(len(COMP))[::-1]
for yi, (x, r) in zip(y, COMP.iterrows()):
    cor = VERMELHO if r.acidentes > r.acidentes_viz else VERDE
    ax1.plot([r.acidentes_viz, r.acidentes], [yi, yi], color=cor, linewidth=2.5, alpha=0.8)
    ax1.plot(r.acidentes_viz, yi, 'o', color=CINZA, markersize=9)
    ax1.plot(r.acidentes, yi, 'o', color=cor, markersize=11)
ax1.set_yticks(y)
ax1.set_yticklabels([f"{x:%d/%m/%Y}" for x in COMP.index], fontsize=10)
ax1.set_xlabel('Acidentes nas rodovias federais no dia', fontsize=12)
ax1.plot([], [], 'o', color=CINZA, label='média das sextas vizinhas')
ax1.plot([], [], 'o', color=VERMELHO, label='sexta-feira 13 com mais acidentes')
ax1.plot([], [], 'o', color=VERDE, label='sexta-feira 13 com menos acidentes')
ax1.legend(loc='lower right', fontsize=10, frameon=False)
ax1.set_title(f'Sexta-feira 13 contra as sextas vizinhas: mais acidentes em {VENCE} de {N13}\n'
              'rodovias federais, 2015 a 2025, sem feriados e sem a pandemia', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE_PRF, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-153-grafico-01-sexta-a-sexta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · Razão de taxas com IC 95%
fig2, ax2 = plt.subplots(figsize=(12, 6))
nomes = list(RR)
y = np.arange(len(nomes))[::-1]
for yi, nome in zip(y, nomes):
    rr, lo, hi, p = RR[nome]
    ax2.plot([lo, hi], [yi, yi], color=DOURADO, linewidth=3.5)
    ax2.plot(rr, yi, 'o', color=DOURADO, markersize=13)
    ax2.text(hi + 0.02, yi, f"{vir(rr, 2)}  [{vir(lo, 2)}; {vir(hi, 2)}]", va='center', fontsize=11)
ax2.axvline(1, color=VERMELHO, linestyle='--', linewidth=1.8)
ax2.text(1, len(nomes) - 0.4, 'igual às outras sextas', color=VERMELHO, ha='center', fontsize=10)
ax2.set_yticks(y)
ax2.set_yticklabels(nomes, fontsize=12)
ax2.set_ylim(-0.6, len(nomes) - 0.2)
ax2.set_xlabel('Razão sexta-feira 13 / sextas vizinhas · IC 95% (quase-Poisson com efeito de trio)', fontsize=11)
ax2.set_title('A assinatura estatística: um pouco menos acidentes, mortes sem diferença clara\n'
              f'{N13} sextas-feiras 13 pareadas com as sextas vizinhas', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE_PRF, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-153-grafico-02-a-razao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Placebo
fig3, axs = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, v, rot in [(axs[0], 'rr_acidentes', 'acidentes'), (axs[1], 'rr_mortos', 'mortos')]:
    cores = [VERMELHO if d == 13 else CINZA for d in PLAC['dia']]
    ax.bar(PLAC['dia'], PLAC[v] - 1, bottom=1, color=cores, width=0.7, alpha=0.9)
    ax.axhline(1, color='#333', linewidth=1.2)
    ax.set_ylabel(f'Razão de {rot}', fontsize=11)
    r = PLAC.loc[PLAC['dia'] == 13, v].iloc[0]
    ax.text(13, r + (0.012 if r >= 1 else -0.012), f"{POS13[v]}º de {len(PLAC)}", ha='center',
            va='bottom' if r >= 1 else 'top', fontsize=11, color=VERMELHO, fontweight='bold')
    ax.set_ylim(min(PLAC[v]) - 0.06, max(PLAC[v]) + 0.06)
axs[1].set_xticks(PLAC['dia'])
axs[1].set_xlabel('Dia do mês em que a sexta-feira caiu', fontsize=12)
axs[0].set_title('O teste placebo: a sexta 13 no meio das sextas de todos os outros dias do mês\n'
                 'cada barra compara as sextas daquele dia com as sextas vizinhas, mesmas exclusões', fontsize=13, pad=14)
plt.figtext(0.5, 0.005, RODAPE_PRF, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.02, 1, 1))
plt.savefig('dia-153-grafico-03-placebo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · A Bolsa
fig4, ax4 = plt.subplots(figsize=(12, 6.5))
xs = np.linspace(0.2, 0.8, 800)
ax4.plot(xs * 100, pout.pdf(xs), color=CINZA, linewidth=2.5, label=f'outras sextas ({mil(len(rout))})')
ax4.fill_between(xs * 100, pout.pdf(xs), color=CINZA, alpha=0.15)
ax4.plot(xs * 100, p13.pdf(xs), color=DOURADO, linewidth=3, label=f'sextas-feiras 13 ({len(r13)})')
ax4.fill_between(xs * 100, p13.pdf(xs), color=DOURADO, alpha=0.25)
ax4.axvline(50, color=VERMELHO, linestyle='--', linewidth=1.5)
ax4.set_xlabel('Proporção de pregões em que o Ibovespa subiu (%)', fontsize=12)
ax4.set_ylabel('Densidade da posterior', fontsize=12)
ax4.legend(fontsize=11, frameon=False, loc='upper left')
ax4.text(0.98, 0.95, f'retorno médio\n{vir(r13.mean(), 2)}% contra {vir(rout.mean(), 2)}%\n'
         f'diferença IC 95% [{vir(DLO, 2)}; {vir(DHI, 2)}]', transform=ax4.transAxes, ha='right', va='top', fontsize=11)
ax4.set_title('Na Bolsa, a sexta-feira 13 sobe um pouco menos de vezes, mas a incerteza engole a diferença\n'
              'Ibovespa 2000-2026, sem março a junho de 2020 · posteriores Beta de Jeffreys', fontsize=12.5, pad=14)
plt.figtext(0.5, 0.01, RODAPE_BOLSA, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-153-grafico-04-a-bolsa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Nas rodovias federais, de 2015 a 2025, a sexta-feira 13 teve 6% menos acidentes do que as
sextas vizinhas. Teve mais acidentes em só 5 de 15 comparações. Entre as sextas dos 31 dias do
mês, ficou em 30º lugar. As mortes foram 14% maiores, mas a faixa vai de 5% a menos até 36% a
mais, e outros dias comuns do mês tiveram números parecidos.**

A sexta-feira 13 não é mais perigosa nas estradas. Se ela tem alguma diferença, é ter um pouco
menos de acidentes, e mesmo isso some quando os feriados voltam para a conta (razão de 0,98).
Talvez porque quem tem medo dirija com mais cuidado ou fique em casa, mas
esses dados não conseguem dizer o porquê.

Na Bolsa, o Ibovespa subiu em 43% das sextas-feiras 13, contra 52% das outras sextas. A diferença
cabe no acaso, e o retorno médio quase não muda.

O azar da sexta-feira 13 aparece nas histórias, não nos registros.

*Qual foi a última coisa que você deixou de fazer por causa de uma data, e o que os números
diriam sobre ela?*

---

### ⚠️ Limitações do Modelo

- Os dados da PRF cobrem só as rodovias federais. Ruas das cidades e estradas estaduais ficam de
  fora.
- O número de acidentes não diz quanta gente estava na estrada. Se quem tem medo da sexta-feira
  13 fica em casa, menos carros poderiam esconder um risco maior por carro. Radun e Summala fazem
  a mesma ressalva.
- A lista de feriados é só a nacional. Feriados estaduais e municipais continuam na conta.
- O corte da pandemia (março a junho de 2020) é uma escolha. Os resultados com e sem ele estão no
  notebook para conferência.
- Com cerca de 15 sextas-feiras 13, o estudo tem força para ver um efeito grande, não um efeito
  pequeno. Um aumento de poucos por cento não pode ser descartado.
- O placebo usa as mesmas regras para os 31 dias do mês. Algum dia vai aparecer no topo por acaso,
  e por isso o ranking não é prova de nada sozinho.
- O retorno da Bolsa é de fechamento a fechamento, sem dividendos. Os achados de outros países
  (Kolb e Rodriguez, Lucey) foram conferidos por resumos e citações, não pelo texto completo.
- Sem ×0,80, porque são registros administrativos e preços de mercado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
